<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03b_feature_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_feature_application**

## **Introducción**

Esta notebook corresponde al stage_03 - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación e importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [3]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [4]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

### 0.3. Definición de rutas



In [5]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [13]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = DRIVE_DIR / Path(os.environ.get("IN_PARQUET", "data/03_targets/mnq_intraday_t2.parquet"))
#IN_RAW_PARQUET = Path(os.environ.get("IN_RAW_PARQUET", "data/01_raw/mnq_intraday.parquet"))
#IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))



### 0.4. Códigos auxiliares para carga de datos y visualización


In [16]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [17]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

# **1. Carga de dataset**

In [18]:
mnq_intraday = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (1024062, 15)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'close_fwd_30', 'delta_30', 'close_fwd_60', 'delta_60', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2026-04-17 16:00:00-04:00
First/Last day: 2020-01-02  ->  2026-04-17
Total days (trading): 1482
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2026-04-17 20:00:00+00:00


### Validación temporal de dataset `mnq_intraday`


In [19]:
import pandas as pd
import numpy as np

# ============================================================
# Validación temporal de mnq_intraday
# Ejecutar inmediatamente después de:
# mnq_intraday = load_mnq_parquet()
# ============================================================

def validate_mnq_intraday(df: pd.DataFrame, verbose: bool = True) -> dict:
    """
    Valida consistencia temporal básica para un dataset intradía.

    Chequeos:
    1) Índice datetime válido
    2) Orden cronológico global
    3) Duplicados de timestamp
    4) Consistencia de columna `date`
    5) Monotonía de `minute_of_day` dentro de cada día
    6) Duplicados de `minute_of_day` dentro de cada día
    7) Saltos temporales negativos o nulos
    8) Gaps intradía distintos de 1 minuto
    """

    result = {
        "ok": True,
        "checks": {},
        "summary": {},
        "artifacts": {}
    }

    df = df.copy()

    # ------------------------------------------------------------
    # 0) Verificaciones básicas de estructura
    # ------------------------------------------------------------
    required_cols = ["date", "minute_of_day", "open", "high", "low", "close", "volume"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser un pd.DatetimeIndex")

    if df.empty:
        raise ValueError("El DataFrame está vacío")

    # ------------------------------------------------------------
    # 1) Orden global del índice
    # ------------------------------------------------------------
    is_monotonic = df.index.is_monotonic_increasing
    has_unique_index = df.index.is_unique
    duplicated_index = df.index[df.index.duplicated()].unique()

    result["checks"]["index_is_monotonic_increasing"] = bool(is_monotonic)
    result["checks"]["index_is_unique"] = bool(has_unique_index)
    result["summary"]["n_duplicated_timestamps"] = int(len(duplicated_index))
    result["artifacts"]["duplicated_timestamps"] = duplicated_index

    # Si no está ordenado, mostramos evidencia pero no reordenamos silenciosamente
    if not is_monotonic:
        diffs_ns = pd.Series(df.index.view("i8")).diff()
        bad_order_pos = np.where(diffs_ns <= 0)[0]
        result["artifacts"]["bad_global_order_positions"] = bad_order_pos[:20]
        result["ok"] = False

    if not has_unique_index:
        result["ok"] = False

    # ------------------------------------------------------------
    # 2) Consistencia entre index.date y columna `date`
    # ------------------------------------------------------------
    # Normalizamos ambos a fecha sin hora
    index_dates = pd.Index(df.index.tz_localize(None).date if df.index.tz is not None else df.index.date)
    col_dates = pd.to_datetime(df["date"]).dt.date

    date_match = (index_dates == col_dates).all()
    result["checks"]["date_column_matches_index_date"] = bool(date_match)

    if not date_match:
        mismatch_mask = index_dates != col_dates
        mismatches = df.loc[mismatch_mask, ["date", "minute_of_day", "close"]].head(20)
        result["artifacts"]["date_mismatches_head"] = mismatches
        result["summary"]["n_date_mismatches"] = int(mismatch_mask.sum())
        result["ok"] = False
    else:
        result["summary"]["n_date_mismatches"] = 0

    # ------------------------------------------------------------
    # 3) Diferencias temporales globales
    # ------------------------------------------------------------
    # Trabajamos en segundos
    diffs_sec = pd.Series(df.index).diff().dt.total_seconds()

    n_non_positive_diffs = int((diffs_sec.iloc[1:] <= 0).sum())
    result["checks"]["all_global_time_diffs_positive"] = (n_non_positive_diffs == 0)
    result["summary"]["n_non_positive_global_diffs"] = n_non_positive_diffs

    if n_non_positive_diffs > 0:
        bad_diff_rows = df.iloc[np.where((diffs_sec <= 0).fillna(False))[0][:20]]
        result["artifacts"]["non_positive_global_diffs_head"] = bad_diff_rows
        result["ok"] = False

    # ------------------------------------------------------------
    # 4) Validación por día
    # ------------------------------------------------------------
    daily_stats = []
    bad_minute_order_days = []
    duplicate_minute_days = []
    intraday_gap_rows = []

    grouped = df.groupby("date", sort=False)

    for day, g in grouped:
        g = g.copy()

        # 4.1 orden del índice dentro del día
        idx_mono = g.index.is_monotonic_increasing

        # 4.2 minute_of_day creciente dentro del día
        mod_diff = g["minute_of_day"].diff()
        minute_order_ok = bool((mod_diff.iloc[1:] > 0).all())

        # 4.3 duplicados de minute_of_day dentro del día
        dup_mod = g["minute_of_day"].duplicated().sum()
        has_dup_mod = dup_mod > 0

        # 4.4 gaps intradía del índice
        idx_diff_sec = pd.Series(g.index).diff().dt.total_seconds()
        gap_mask = (~idx_diff_sec.isna()) & (idx_diff_sec != 60)

        n_intraday_gaps = int(gap_mask.sum())

        if not minute_order_ok:
            bad_minute_order_days.append(day)

        if has_dup_mod:
            duplicate_minute_days.append(day)

        if n_intraday_gaps > 0:
            gap_info = g.loc[gap_mask, ["date", "minute_of_day", "open", "high", "low", "close", "volume"]].copy()
            gap_info["gap_seconds"] = idx_diff_sec[gap_mask].values
            intraday_gap_rows.append(gap_info)

        daily_stats.append({
            "date": day,
            "n_rows": len(g),
            "index_monotonic": bool(idx_mono),
            "minute_of_day_monotonic": minute_order_ok,
            "n_duplicate_minute_of_day": int(dup_mod),
            "n_intraday_gaps_not_60s": n_intraday_gaps,
            "minute_min": int(g["minute_of_day"].min()),
            "minute_max": int(g["minute_of_day"].max()),
        })

        if not idx_mono or not minute_order_ok or has_dup_mod:
            result["ok"] = False

    daily_stats_df = pd.DataFrame(daily_stats)

    result["artifacts"]["daily_stats"] = daily_stats_df
    result["summary"]["n_days"] = int(daily_stats_df.shape[0])
    result["summary"]["days_with_bad_minute_order"] = int(len(bad_minute_order_days))
    result["summary"]["days_with_duplicate_minute_of_day"] = int(len(duplicate_minute_days))
    result["summary"]["days_with_intraday_gaps_not_60s"] = int((daily_stats_df["n_intraday_gaps_not_60s"] > 0).sum())

    result["checks"]["all_days_have_monotonic_minute_of_day"] = (len(bad_minute_order_days) == 0)
    result["checks"]["no_duplicate_minute_of_day_within_day"] = (len(duplicate_minute_days) == 0)
    result["checks"]["all_intraday_steps_are_60s_within_day"] = bool(
        (daily_stats_df["n_intraday_gaps_not_60s"] == 0).all()
    )

    result["artifacts"]["bad_minute_order_days"] = bad_minute_order_days
    result["artifacts"]["duplicate_minute_days"] = duplicate_minute_days

    if intraday_gap_rows:
        result["artifacts"]["intraday_gaps_head"] = pd.concat(intraday_gap_rows, axis=0).head(50)
    else:
        result["artifacts"]["intraday_gaps_head"] = pd.DataFrame()

    # ------------------------------------------------------------
    # 5) Resumen global
    # ------------------------------------------------------------
    result["summary"]["n_rows"] = int(len(df))
    result["summary"]["start"] = df.index.min()
    result["summary"]["end"] = df.index.max()

    # ------------------------------------------------------------
    # 6) Reporte por pantalla
    # ------------------------------------------------------------
    if verbose:
        print("=" * 70)
        print("VALIDACIÓN TEMPORAL DE mnq_intraday")
        print("=" * 70)
        print(f"Rows                     : {result['summary']['n_rows']}")
        print(f"Days                     : {result['summary']['n_days']}")
        print(f"Start                    : {result['summary']['start']}")
        print(f"End                      : {result['summary']['end']}")
        print("-" * 70)
        print(f"Index monotonic          : {result['checks']['index_is_monotonic_increasing']}")
        print(f"Index unique             : {result['checks']['index_is_unique']}")
        print(f"Date == index.date       : {result['checks']['date_column_matches_index_date']}")
        print(f"Global diffs > 0         : {result['checks']['all_global_time_diffs_positive']}")
        print(f"minute_of_day monotonic  : {result['checks']['all_days_have_monotonic_minute_of_day']}")
        print(f"No dup minute_of_day     : {result['checks']['no_duplicate_minute_of_day_within_day']}")
        print(f"Intraday steps = 60s     : {result['checks']['all_intraday_steps_are_60s_within_day']}")
        print("-" * 70)
        print(f"Duplicated timestamps    : {result['summary']['n_duplicated_timestamps']}")
        print(f"Date mismatches          : {result['summary']['n_date_mismatches']}")
        print(f"Non-positive global diffs: {result['summary']['n_non_positive_global_diffs']}")
        print(f"Bad minute order days    : {result['summary']['days_with_bad_minute_order']}")
        print(f"Dup minute_of_day days   : {result['summary']['days_with_duplicate_minute_of_day']}")
        print(f"Days with !=60s gaps     : {result['summary']['days_with_intraday_gaps_not_60s']}")
        print("-" * 70)
        print(f"DATASET OK               : {result['ok']}")
        print("=" * 70)

        if result["summary"]["n_duplicated_timestamps"] > 0:
            print("\nDuplicated timestamps (head):")
            print(pd.Index(result["artifacts"]["duplicated_timestamps"][:10]))

        if result["summary"]["n_date_mismatches"] > 0:
            print("\nDate mismatches (head):")
            print(result["artifacts"]["date_mismatches_head"])

        if result["summary"]["days_with_bad_minute_order"] > 0:
            print("\nDays with bad minute_of_day order (head):")
            print(result["artifacts"]["bad_minute_order_days"][:10])

        if result["summary"]["days_with_duplicate_minute_of_day"] > 0:
            print("\nDays with duplicate minute_of_day (head):")
            print(result["artifacts"]["duplicate_minute_days"][:10])

        if not result["artifacts"]["intraday_gaps_head"].empty:
            print("\nIntraday gaps != 60 seconds (head):")
            print(result["artifacts"]["intraday_gaps_head"])

    return result


# ============================================================
# Ejecución inmediata
# ============================================================
validation = validate_mnq_intraday(mnq_intraday, verbose=True)

# Si quiere abortar automáticamente cuando haya problemas críticos:
critical_checks = [
    "index_is_monotonic_increasing",
    "index_is_unique",
    "date_column_matches_index_date",
    "all_global_time_diffs_positive",
    "all_days_have_monotonic_minute_of_day",
    "no_duplicate_minute_of_day_within_day",
]

failed_critical = [k for k in critical_checks if not validation["checks"].get(k, False)]

if failed_critical:
    raise ValueError(
        "Validación temporal fallida. Checks críticos con error: "
        + ", ".join(failed_critical)
    )

VALIDACIÓN TEMPORAL DE mnq_intraday
Rows                     : 1024062
Days                     : 1482
Start                    : 2020-01-02 04:30:00-05:00
End                      : 2026-04-17 16:00:00-04:00
----------------------------------------------------------------------
Index monotonic          : True
Index unique             : True
Date == index.date       : True
Global diffs > 0         : True
minute_of_day monotonic  : True
No dup minute_of_day     : True
Intraday steps = 60s     : True
----------------------------------------------------------------------
Duplicated timestamps    : 0
Date mismatches          : 0
Non-positive global diffs: 0
Bad minute order days    : 0
Dup minute_of_day days   : 0
Days with !=60s gaps     : 0
----------------------------------------------------------------------
DATASET OK               : True


In [ ]:
't2_p40_h30', 't2_p40_h60', 't2_p50_h30'],

#**2. Agregado de variables temporaltes**

In [21]:
import numpy as np
import pandas as pd


def build_intraday_time_structure_pipeline(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    date_col: str = "date",
    regime_col: str = "regime_id",
    sort_index_if_needed: bool = True,
    drop_duplicate_index: bool = False,
    verbose: bool = True,
):
    """
    Pipeline para construir la estructura temporal intradía mínima del dataset.

    Este pipeline:
    1) valida que el índice sea DatetimeIndex
    2) ordena cronológicamente por índice si es necesario
    3) opcionalmente elimina índices duplicados
    4) agrega la columna minute_of_day
    5) agrega la columna date
    6) agrega la columna regime_id

    Codificación de regime_id
    -------------------------
    0 = overnight
    1 = premarket
    2 = opening
    3 = regular
    4 = closing

    Regímenes intradía (hora NY, intervalos [inicio, fin))
    ------------------------------------------------------
    - premarket : 08:30 <= t < 09:30
    - opening   : 09:30 <= t < 10:30
    - regular   : 10:30 <= t < 15:30
    - closing   : 15:30 <= t < 16:00
    - overnight : resto

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada con índice datetime.
    minute_col : str, default="minute_of_day"
        Nombre de la columna de minuto del día.
    date_col : str, default="date"
        Nombre de la columna de fecha.
    regime_col : str, default="regime_id"
        Nombre de la columna de régimen.
    sort_index_if_needed : bool, default=True
        Si True, ordena el DataFrame por índice si no está ordenado.
    drop_duplicate_index : bool, default=False
        Si True, elimina duplicados de índice conservando la primera ocurrencia.
    verbose : bool, default=True
        Si True, imprime reporte final.

    Retorna
    -------
    out : pd.DataFrame
        DataFrame con columnas agregadas.
    summary : dict
        Resumen estructurado del proceso.
    """

    regime_name_map = {
        0: "overnight",
        1: "premarket",
        2: "opening",
        3: "regular",
        4: "closing",
    }

    # ---------------------------------------------------------------------
    # Subfunción 1: validaciones básicas
    # ---------------------------------------------------------------------
    def _validate_input(data: pd.DataFrame) -> None:
        if data.empty:
            raise ValueError("El DataFrame está vacío.")

        if not isinstance(data.index, pd.DatetimeIndex):
            raise TypeError("El DataFrame debe tener un DatetimeIndex.")

    # ---------------------------------------------------------------------
    # Subfunción 2: normalización y orden cronológico
    # ---------------------------------------------------------------------
    def _normalize_index(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()

        # Asegurar DatetimeIndex consistente
        out_local.index = pd.to_datetime(out_local.index)

        # Orden cronológico
        if not out_local.index.is_monotonic_increasing:
            if sort_index_if_needed:
                out_local = out_local.sort_index().copy()
            else:
                raise ValueError("El índice datetime no está ordenado crecientemente.")

        # Duplicados de índice
        n_dup_idx = int(out_local.index.duplicated().sum())
        if n_dup_idx > 0:
            if drop_duplicate_index:
                out_local = out_local.loc[~out_local.index.duplicated(keep="first")].copy()
            else:
                raise ValueError(
                    f"Se encontraron {n_dup_idx} timestamps duplicados en el índice."
                )

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 3: minute_of_day
    # ---------------------------------------------------------------------
    def _add_minute_of_day(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()
        out_local[minute_col] = out_local.index.hour * 60 + out_local.index.minute
        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 4: date
    # ---------------------------------------------------------------------
    def _add_date_column(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()
        out_local[date_col] = out_local.index.date

        # Llevar date al frente
        front_cols = [date_col]
        other_cols = [c for c in out_local.columns if c not in front_cols]
        out_local = out_local[front_cols + other_cols]

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 5: regime_id
    # ---------------------------------------------------------------------
    def _add_market_regime(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()

        if minute_col not in out_local.columns:
            raise ValueError(f"No existe la columna {minute_col}.")

        m = out_local[minute_col]

        out_local[regime_col] = np.select(
            [
                (m >= 510) & (m < 570),   # premarket
                (m >= 570) & (m < 630),   # opening
                (m >= 630) & (m < 930),   # regular
                (m >= 930) & (m < 960),   # closing
            ],
            [
                1,  # premarket
                2,  # opening
                3,  # regular
                4,  # closing
            ],
            default=0,  # overnight
        ).astype(np.int8)

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 6: validaciones finales del pipeline
    # ---------------------------------------------------------------------
    def _validate_output(data: pd.DataFrame) -> None:
        required_cols = [date_col, minute_col, regime_col]
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"Faltan columnas de salida esperadas: {missing}")

        if not data.index.is_monotonic_increasing:
            raise ValueError("El output final no quedó ordenado cronológicamente.")

        if data[minute_col].isna().any():
            raise ValueError(f"La columna {minute_col} contiene NaN.")

        if data[regime_col].isna().any():
            raise ValueError(f"La columna {regime_col} contiene NaN.")

        invalid_minutes = (~data[minute_col].between(0, 1439)).sum()
        if invalid_minutes > 0:
            raise ValueError(
                f"Se encontraron {invalid_minutes} valores inválidos en {minute_col}."
            )

        valid_regimes = {0, 1, 2, 3, 4}
        invalid_regimes = (~data[regime_col].isin(valid_regimes)).sum()
        if invalid_regimes > 0:
            raise ValueError(
                f"Se encontraron {invalid_regimes} valores inválidos en {regime_col}."
            )

    # ---------------------------------------------------------------------
    # Subfunción 7: resumen
    # ---------------------------------------------------------------------
    def _build_summary(original_df: pd.DataFrame, final_df: pd.DataFrame) -> dict:
        regime_counts = final_df[regime_col].value_counts(dropna=False).sort_index().to_dict()
        regime_distribution = {}

        total_rows = len(final_df)

        for rid in sorted(regime_counts.keys()):
            count = int(regime_counts[rid])
            regime_distribution[int(rid)] = {
                "name": regime_name_map.get(int(rid), "unknown"),
                "count": count,
                "pct": float(count / total_rows) if total_rows > 0 else np.nan,
            }

        summary_local = {
            "n_rows_input": int(len(original_df)),
            "n_rows_output": int(len(final_df)),
            "n_columns_output": int(final_df.shape[1]),
            "index_is_datetime": isinstance(final_df.index, pd.DatetimeIndex),
            "index_is_sorted": bool(final_df.index.is_monotonic_increasing),
            "n_duplicate_index": int(final_df.index.duplicated().sum()),
            "n_sessions": int(final_df[date_col].nunique()),
            "first_timestamp": str(final_df.index.min()),
            "last_timestamp": str(final_df.index.max()),
            "minute_col": minute_col,
            "date_col": date_col,
            "regime_col": regime_col,
            "minute_min": int(final_df[minute_col].min()),
            "minute_max": int(final_df[minute_col].max()),
            "regime_distribution": regime_distribution,
        }

        return summary_local

    # ---------------------------------------------------------------------
    # Subfunción 8: reporte
    # ---------------------------------------------------------------------
    def _print_report(summary_local: dict) -> None:
        print("=" * 100)
        print("REPORTE | INTRADAY TIME STRUCTURE PIPELINE")
        print("=" * 100)
        print(f"Filas de entrada         : {summary_local['n_rows_input']:,}")
        print(f"Filas de salida          : {summary_local['n_rows_output']:,}")
        print(f"Columnas de salida       : {summary_local['n_columns_output']:,}")
        print(f"Índice datetime          : {summary_local['index_is_datetime']}")
        print(f"Índice ordenado          : {summary_local['index_is_sorted']}")
        print(f"Duplicados en índice     : {summary_local['n_duplicate_index']:,}")
        print(f"Sesiones únicas          : {summary_local['n_sessions']:,}")
        print(f"Primer timestamp         : {summary_local['first_timestamp']}")
        print(f"Último timestamp         : {summary_local['last_timestamp']}")
        print(f"Rango {summary_local['minute_col']}     : {summary_local['minute_min']} - {summary_local['minute_max']}")

        print("-" * 100)
        print("DISTRIBUCIÓN DE REGÍMENES")
        print("-" * 100)

        for rid, info in summary_local["regime_distribution"].items():
            print(
                f"regime_id = {rid} | "
                f"{info['name']:<10} | "
                f"count = {info['count']:,} | "
                f"pct = {info['pct']:.2%}"
            )

        print("=" * 100)

    # ---------------------------------------------------------------------
    # Ejecución principal
    # ---------------------------------------------------------------------
    _validate_input(df)

    out = _normalize_index(df)
    out = _add_minute_of_day(out)
    out = _add_date_column(out)
    out = _add_market_regime(out)

    _validate_output(out)

    summary = _build_summary(df, out)

    if verbose:
        _print_report(summary)

    return out, summary

In [22]:
mnq_intraday, time_summary = build_intraday_time_structure_pipeline(
    mnq_intraday,
    minute_col="minute_of_day",
    date_col="date",
    regime_col="regime_id",
    sort_index_if_needed=True,
    drop_duplicate_index=False,
    verbose=True,
)

REPORTE | INTRADAY TIME STRUCTURE PIPELINE
Filas de entrada         : 1,024,062
Filas de salida          : 1,024,062
Columnas de salida       : 15
Índice datetime          : True
Índice ordenado          : True
Duplicados en índice     : 0
Sesiones únicas          : 1,482
Primer timestamp         : 2020-01-02 04:30:00-05:00
Último timestamp         : 2026-04-17 16:00:00-04:00
Rango minute_of_day     : 270 - 960
----------------------------------------------------------------------------------------------------
DISTRIBUCIÓN DE REGÍMENES
----------------------------------------------------------------------------------------------------
regime_id = 0 | overnight  | count = 357,162 | pct = 34.88%
regime_id = 1 | premarket  | count = 88,920 | pct = 8.68%
regime_id = 2 | opening    | count = 88,920 | pct = 8.68%
regime_id = 3 | regular    | count = 444,600 | pct = 43.42%
regime_id = 4 | closing    | count = 44,460 | pct = 4.34%


In [23]:
mnq_intraday

,date,open,high,low,close,volume,minute_of_day,regime_id,close_fwd_30,delta_30,close_fwd_60,delta_60,t2_p40_h30,t2_p40_h60,t2_p50_h30
datetime,,,,,,,,,,,,,,,
2020-01-02 04:30:00-05:00,2020-01-02,8813.25,8813.25,8812.50,8813.25,13,270,0,8815.75,2.50,8819.25,6.00,NaN,NaN,NaN
2020-01-02 04:31:00-05:00,2020-01-02,8812.50,8812.50,8811.00,8812.50,121,271,0,8815.75,3.25,8818.75,6.25,NaN,NaN,NaN
2020-01-02 04:32:00-05:00,2020-01-02,8812.50,8813.25,8811.75,8811.75,53,272,0,8817.50,5.75,8819.00,7.25,NaN,NaN,NaN
2020-01-02 04:33:00-05:00,2020-01-02,8812.00,8812.25,8810.50,8810.50,37,273,0,8816.00,5.50,8818.25,7.75,NaN,NaN,NaN
2020-01-02 04:34:00-05:00,2020-01-02,8810.75,8812.25,8810.50,8812.00,36,274,0,8816.00,4.00,8818.75,6.75,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-17 15:56:00-04:00,2026-04-17,26815.50,26829.00,26814.25,26820.00,4185,956,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-17 15:57:00-04:00,2026-04-17,26819.75,26830.00,26815.00,26817.50,2536,957,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-17 15:58:00-04:00,2026-04-17,26818.00,26824.50,26809.00,26823.75,2518,958,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **3. Indicadores Técnicos**

## **3.1. Función para cálculo de Indicadores técnicos**

In [24]:
features_core = [
    "roc_60",
    "ema_60",
    "roc_30",
    "stoch_k_30",
    "regime_id",
]

features_extended = [
    "roc_60",
    "ema_60",
    "roc_30",
    "stoch_k_30",
    "mom_5",
    "atr_norm_10",
    "macd",
    "regime_id",
]


In [25]:
import numpy as np
import pandas as pd

from ta.momentum import ROCIndicator, StochasticOscillator
from ta.volatility import AverageTrueRange
from ta.trend import MACD


def build_technical_indicators_pipeline(
    df: pd.DataFrame,
    *,
    target: str = "close",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    high_col: str = "high",
    low_col: str = "low",
    validate_order: bool = True,
    drop_na_indicator_rows: bool = False,
    verbose: bool = True,
):
    """
    Pipeline completo para calcular indicadores técnicos intradía por jornada.

    Indicadores calculados
    ----------------------
    - roc_30
    - roc_60
    - stoch_k_30
    - atr_norm_10
    - ema_60
    - mom_5
    - macd

    Flujo
    -----
    1) Valida columnas requeridas
    2) Verifica que el índice sea DatetimeIndex
    3) Verifica orden cronológico global
    4) Verifica orden cronológico dentro de cada jornada
    5) Calcula indicadores por día
    6) Imprime reporte final

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame intradía de entrada.
    target : str, default="close"
        Columna objetivo sobre la cual se calculan ROC y Stochastic.
    date_col : str, default="date"
        Columna que identifica la jornada.
    minute_col : str, default="minute_of_day"
        Columna de orden intradía.
    high_col : str, default="high"
        Columna high.
    low_col : str, default="low"
        Columna low.
    validate_order : bool, default=True
        Si True, verifica orden cronológico y duplicados.
    drop_na_indicator_rows : bool, default=False
        Si True, elimina filas con NaN en cualquiera de los indicadores calculados.
    verbose : bool, default=True
        Si True, imprime reporte final.

    Retorna
    -------
    out : pd.DataFrame
        DataFrame con indicadores agregados.
    summary : dict
        Resumen estructurado del proceso.
    """

    indicator_cols = [
        "roc_30",
        "roc_60",
        "stoch_k_30",
        "atr_norm_10",
        "ema_60",
        "mom_5",
        "macd",
    ]

    # ---------------------------------------------------------------------
    # Subfunción 1: validaciones estructurales
    # ---------------------------------------------------------------------
    def _validate_input(data: pd.DataFrame) -> None:
        required_cols = [target, high_col, low_col, date_col, minute_col]
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"Faltan columnas requeridas: {missing}")

        if data.empty:
            raise ValueError("El DataFrame está vacío")

        if not isinstance(data.index, pd.DatetimeIndex):
            raise TypeError("El índice debe ser un pd.DatetimeIndex")

    # ---------------------------------------------------------------------
    # Subfunción 2: validación de orden cronológico global y por jornada
    # ---------------------------------------------------------------------
    def _validate_chronological_order(data: pd.DataFrame) -> None:
        if not data.index.is_monotonic_increasing:
            raise ValueError("El índice datetime no está ordenado crecientemente")

        sorted_index = data.sort_values([date_col, minute_col]).index
        if not data.index.equals(sorted_index):
            raise ValueError(
                f"El DataFrame no está ordenado por [{date_col}, {minute_col}]."
            )

        n_dupes = data.duplicated(subset=[date_col, minute_col]).sum()
        if n_dupes > 0:
            raise ValueError(
                f"Se encontraron {n_dupes} combinaciones duplicadas de "
                f"[{date_col}, {minute_col}]."
            )

        # Verificación adicional por jornada
        for session_date, g in data.groupby(date_col, sort=False):
            if not g.index.is_monotonic_increasing:
                raise ValueError(
                    f"La jornada {session_date} no está ordenada cronológicamente por índice."
                )

            if not g[minute_col].is_monotonic_increasing:
                raise ValueError(
                    f"La jornada {session_date} no está ordenada crecientemente por {minute_col}."
                )

    # ---------------------------------------------------------------------
    # Subfunción 3: cálculo de indicadores por jornada
    # ---------------------------------------------------------------------
    def _apply_indicators_one_day(group: pd.DataFrame) -> pd.DataFrame:
        g = group.copy()

        # ROC 30
        g["roc_30"] = ROCIndicator(
            close=g[target],
            window=30,
        ).roc()

        # ROC 60
        g["roc_60"] = ROCIndicator(
            close=g[target],
            window=60,
        ).roc()

        # Stochastic %K con ventana 30
        stoch_30 = StochasticOscillator(
            high=g[high_col],
            low=g[low_col],
            close=g[target],
            window=30,
            smooth_window=3,
        )
        g["stoch_k_30"] = stoch_30.stoch()

        # ATR normalizado 10
        atr_10 = AverageTrueRange(
            high=g[high_col],
            low=g[low_col],
            close=g[target],
            window=10,
        ).average_true_range()
        g["atr_norm_10"] = atr_10 / g[target]

        # EMA 60
        g["ema_60"] = g[target] / g[target].ewm(span=60, adjust=False).mean() - 1

        # Momentum 5
        g["mom_5"] = g[target].pct_change(5)

        # MACD
        g["macd"] = MACD(close=g[target]).macd_diff()

        return g

    # ---------------------------------------------------------------------
    # Ejecución principal
    # ---------------------------------------------------------------------
    _validate_input(df)

    if validate_order:
        _validate_chronological_order(df)

    out = df.groupby(date_col, group_keys=False).apply(_apply_indicators_one_day)

    if drop_na_indicator_rows:
        out = out.dropna(subset=indicator_cols).copy()

    summary = {
        "n_rows_input": len(df),
        "n_rows_output": len(out),
        "indicator_cols": indicator_cols,
        "n_indicator_cols": len(indicator_cols),
        "drop_na_indicator_rows": drop_na_indicator_rows,
        "validate_order": validate_order,
    }

    if verbose:
        print("=" * 80)
        print("PIPELINE DE INDICADORES TÉCNICOS")
        print("=" * 80)
        print(f"Filas entrada : {summary['n_rows_input']:,}")
        print(f"Filas salida  : {summary['n_rows_output']:,}")
        print(f"Indicadores   : {summary['indicator_cols']}")
        print(f"N indicadores : {summary['n_indicator_cols']}")
        print("=" * 80)

    return out, summary

## **3.2. Cálculo de indicadores técnicos**

In [26]:
mnq_features, tech_summary = build_technical_indicators_pipeline(
    mnq_intraday,
    target="close",
    date_col="date",
    minute_col="minute_of_day",
    high_col="high",
    low_col="low",
    validate_order=True,
    drop_na_indicator_rows=True,
    verbose=True,
)

PIPELINE DE INDICADORES TÉCNICOS
Filas entrada : 1,024,062
Filas salida  : 935,142
Indicadores   : ['roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10', 'ema_60', 'mom_5', 'macd']
N indicadores : 7


# **4. Filtrado por regime_id**

In [34]:
df_r12 = mnq_features[mnq_features["regime_id"].isin([1, 2])].copy()
print(df_r12["regime_id"].value_counts())

regime_id
1    88920
2    88920
Name: count, dtype: int64


In [35]:
target = "t2_p40_h60"

print("NaN en target:", df_r12[target].isna().sum())

NaN en target: 0


In [40]:
mnq_model = df_r12.dropna(subset=[target]).copy()

In [41]:
mnq_model

,date,open,high,low,close,volume,minute_of_day,regime_id,close_fwd_30,delta_30,...,t2_p40_h30,t2_p40_h60,t2_p50_h30,roc_30,roc_60,stoch_k_30,atr_norm_10,ema_60,mom_5,macd
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-02 08:30:00-05:00,2020-01-02,8823.25,8823.50,8821.50,8822.50,61,510,1,8828.25,5.75,...,0.0,0.0,0.0,0.045359,0.090760,66.666667,0.000138,0.000381,-0.000227,-0.089892
2020-01-02 08:31:00-05:00,2020-01-02,8822.75,8823.75,8822.00,8822.25,86,511,1,8828.00,5.75,...,0.0,0.0,0.0,0.039688,0.090762,62.962963,0.000144,0.000341,-0.000170,-0.164868
2020-01-02 08:32:00-05:00,2020-01-02,8823.00,8823.75,8822.50,8823.75,42,512,1,8826.00,2.25,...,0.0,0.0,0.0,0.042517,0.096424,85.185185,0.000147,0.000495,0.000028,-0.118071
2020-01-02 08:33:00-05:00,2020-01-02,8824.25,8825.75,8823.50,8825.75,134,513,1,8822.00,-3.75,...,0.0,0.0,0.0,0.079376,0.121951,100.000000,0.000157,0.000698,0.000198,0.034776
2020-01-02 08:34:00-05:00,2020-01-02,8826.00,8826.50,8825.25,8826.00,102,514,1,8823.50,-2.50,...,0.0,0.0,0.0,0.085048,0.113430,93.939394,0.000156,0.000702,0.000368,0.134320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-17 10:25:00-04:00,2026-04-17,26732.00,26743.75,26727.00,26738.75,3655,625,2,26838.25,99.50,...,1.0,1.0,1.0,0.102016,-0.044859,58.309859,0.000768,0.000304,-0.000271,-4.477908
2026-04-17 10:26:00-04:00,2026-04-17,26739.25,26744.75,26733.50,26738.50,3076,626,2,26804.75,66.25,...,1.0,1.0,1.0,0.135756,-0.000935,58.028169,0.000733,0.000285,-0.000103,-4.186233
2026-04-17 10:27:00-04:00,2026-04-17,26738.50,26748.50,26729.50,26733.50,4018,627,2,26832.75,99.25,...,1.0,1.0,1.0,0.108596,0.060822,49.702381,0.000731,0.000095,-0.000645,-4.152592


# **5. Verificar existencia de señal**

## **5.1. Creación de split temporales**

In [42]:
import pandas as pd


def time_based_split_by_date(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    train_ratio: float = 0.70,
    valid_ratio: float = 0.15,
    test_ratio: float = 0.15,
    verbose: bool = True,
):
    """
    Realiza un split temporal (train/valid/test) basado en fechas.

    - No mezcla días
    - Respeta orden cronológico
    - Evita leakage

    Retorna
    -------
    train_df, valid_df, test_df
    """

    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-6:
        raise ValueError("Los ratios deben sumar 1.0")

    # Ordenar por fecha y minuto (por seguridad)
    df_sorted = df.sort_values([date_col, "minute_of_day"]).copy()

    # Fechas únicas ordenadas
    unique_dates = df_sorted[date_col].drop_duplicates().sort_values().values
    n_dates = len(unique_dates)

    # Índices de corte
    train_end = int(n_dates * train_ratio)
    valid_end = train_end + int(n_dates * valid_ratio)

    # Fechas por split
    train_dates = unique_dates[:train_end]
    valid_dates = unique_dates[train_end:valid_end]
    test_dates  = unique_dates[valid_end:]

    # Crear splits
    train_df = df_sorted[df_sorted[date_col].isin(train_dates)].copy()
    valid_df = df_sorted[df_sorted[date_col].isin(valid_dates)].copy()
    test_df  = df_sorted[df_sorted[date_col].isin(test_dates)].copy()

    if verbose:
        print("=" * 80)
        print("TIME SPLIT SUMMARY")
        print("=" * 80)

        print(f"Total días        : {n_dates}")
        print(f"Train días        : {len(train_dates)} ({len(train_dates)/n_dates:.2%})")
        print(f"Valid días        : {len(valid_dates)} ({len(valid_dates)/n_dates:.2%})")
        print(f"Test días         : {len(test_dates)} ({len(test_dates)/n_dates:.2%})")

        print("-" * 80)

        print(f"Train fechas      : {train_dates[0]} → {train_dates[-1]}")
        print(f"Valid fechas      : {valid_dates[0]} → {valid_dates[-1]}")
        print(f"Test fechas       : {test_dates[0]} → {test_dates[-1]}")

        print("-" * 80)

        print(f"Train filas       : {len(train_df):,}")
        print(f"Valid filas       : {len(valid_df):,}")
        print(f"Test filas        : {len(test_df):,}")

        print("=" * 80)

    return train_df, valid_df, test_df

In [44]:
mnq_train, mnq_valid, mnq_test = time_based_split_by_date(
    mnq_model,
    date_col="date",
    train_ratio=0.70,
    valid_ratio=0.15,
    test_ratio=0.15,
    verbose=True,
)

assert mnq_train["date"].max() < mnq_valid["date"].min()
assert mnq_valid["date"].max() < mnq_test["date"].min()

TIME SPLIT SUMMARY
Total días        : 1482
Train días        : 1037 (69.97%)
Valid días        : 222 (14.98%)
Test días         : 223 (15.05%)
--------------------------------------------------------------------------------
Train fechas      : 2020-01-02 → 2024-05-13
Valid fechas      : 2024-05-14 → 2025-05-06
Test fechas       : 2025-05-07 → 2026-04-17
--------------------------------------------------------------------------------
Train filas       : 124,440
Valid filas       : 26,640
Test filas        : 26,760


### Verificación de balance de clases

In [47]:
import pandas as pd
import numpy as np

# ============================================================
# Configuración
# ============================================================
targets = [
    "t2_p40_h30",
    "t2_p40_h60",
    "t2_p50_h30",
]

# ============================================================
# Análisis base de existencia de señal
# ============================================================
rows = []

for target in targets:
    df = mnq_train.dropna(subset=[target]).copy()

    # Conteos base
    n_total = len(df)
    counts = df[target].value_counts(dropna=False).to_dict()

    n_short = counts.get(-1, 0)
    n_flat  = counts.get(0, 0)
    n_long  = counts.get(1, 0)

    pct_short = n_short / n_total if n_total > 0 else np.nan
    pct_flat  = n_flat  / n_total if n_total > 0 else np.nan
    pct_long  = n_long  / n_total if n_total > 0 else np.nan

    # Operabilidad
    signal_mask = df[target] != 0
    n_signals = signal_mask.sum()
    signal_rate = n_signals / n_total if n_total > 0 else np.nan

    # Consistencia diaria
    daily_signal_rate = (
        df.assign(signal=signal_mask)
          .groupby("date")["signal"]
          .mean()
    )

    daily_days = daily_signal_rate.shape[0]
    days_with_signal = (daily_signal_rate > 0).sum()
    pct_days_with_signal = days_with_signal / daily_days if daily_days > 0 else np.nan

    daily_mean = daily_signal_rate.mean()
    daily_std = daily_signal_rate.std()
    daily_min = daily_signal_rate.min()
    daily_p25 = daily_signal_rate.quantile(0.25)
    daily_median = daily_signal_rate.median()
    daily_p75 = daily_signal_rate.quantile(0.75)
    daily_max = daily_signal_rate.max()
    daily_cv = daily_std / daily_mean if pd.notna(daily_mean) and daily_mean != 0 else np.nan

    # Heurísticas simples
    class_balance_ok = (
        (0.40 <= pct_flat <= 0.60) and
        (pct_short >= 0.20) and
        (pct_long >= 0.20)
    )

    signal_rate_ok = 0.30 <= signal_rate <= 0.60
    signal_rate_band = (
        "muy bajo" if signal_rate < 0.20 else
        "bajo" if signal_rate < 0.30 else
        "interesante" if signal_rate <= 0.60 else
        "alto/ruidoso"
    )

    rows.append({
        "target": target,
        "n_total": n_total,
        "n_short": n_short,
        "n_flat": n_flat,
        "n_long": n_long,
        "pct_short": pct_short,
        "pct_flat": pct_flat,
        "pct_long": pct_long,
        "n_signals": n_signals,
        "signal_rate": signal_rate,
        "signal_rate_band": signal_rate_band,
        "class_balance_ok": class_balance_ok,
        "signal_rate_ok": signal_rate_ok,
        "days": daily_days,
        "days_with_signal": days_with_signal,
        "pct_days_with_signal": pct_days_with_signal,
        "daily_signal_mean": daily_mean,
        "daily_signal_std": daily_std,
        "daily_signal_cv": daily_cv,
        "daily_signal_min": daily_min,
        "daily_signal_p25": daily_p25,
        "daily_signal_median": daily_median,
        "daily_signal_p75": daily_p75,
        "daily_signal_max": daily_max,
    })

summary_signal = pd.DataFrame(rows).sort_values(
    ["class_balance_ok", "signal_rate_ok", "signal_rate"],
    ascending=[False, False, False]
).reset_index(drop=True)

# ============================================================
# Mostrar resumen principal
# ============================================================
pd.set_option("display.max_columns", None)
print("=" * 120)
print("RESUMEN DE EXISTENCIA DE SEÑAL - TRAIN")
print("=" * 120)
print(summary_signal)

# ============================================================
# Mostrar distribución por clase más legible
# ============================================================
print("\n" + "=" * 120)
print("DISTRIBUCIÓN DE CLASES")
print("=" * 120)
print(
    summary_signal[
        [
            "target",
            "pct_short",
            "pct_flat",
            "pct_long",
            "signal_rate",
            "signal_rate_band",
            "pct_days_with_signal",
            "daily_signal_mean",
            "daily_signal_std",
            "daily_signal_cv",
        ]
    ]
    .to_string(index=False, float_format=lambda x: f"{x:.4f}")
)

RESUMEN DE EXISTENCIA DE SEÑAL - TRAIN
       target  n_total  n_short  n_flat  n_long  pct_short  pct_flat  \
0  t2_p40_h30   124440    33827   53801   36812   0.271834  0.432345   
1  t2_p40_h60   124440    33605   54061   36774   0.270050  0.434434   
2  t2_p50_h30   124440    28029   66319   30092   0.225241  0.532940   

   pct_long  n_signals  signal_rate signal_rate_band  class_balance_ok  \
0  0.295821      70639     0.567655      interesante              True   
1  0.295516      70379     0.565566      interesante              True   
2  0.241819      58121     0.467060      interesante              True   

   signal_rate_ok  days  days_with_signal  pct_days_with_signal  \
0            True  1037              1036              0.999036   
1            True  1037              1026              0.989392   
2            True  1037              1030              0.993250   

   daily_signal_mean  daily_signal_std  daily_signal_cv  daily_signal_min  \
0           0.567655         

**Análisis resumido de existencia de señal (targets T2)**

1. **Señal presente y operable**

   * `signal_rate`: 0.46 – 0.57 (rango adecuado)
   * ~99% de los días con señales
     → Los tres targets son operables

2. **Buen balance de clases**

   * `pct_flat`: 43% – 53%
   * `pct_short` y `pct_long`: > 22%
     → Estructura equilibrada, sin desbalance crítico

3. **Comparación entre targets**

   * `t2_p40_h30`: mayor consistencia (menor variabilidad)
   * `t2_p40_h60`: similar frecuencia, pero más inestable
   * `t2_p50_h30`: más conservador (menos señales, potencialmente mayor calidad)

4. **Consistencia diaria**

   * Mediana diaria: ~0.48 – 0.60
   * Percentil 75: ~0.60 – 0.75
     → Señal estable a lo largo del tiempo

5. **Trade-off clave**

   * `p40`: mayor frecuencia (más trades)
   * `p50`: mayor filtrado (menos ruido)
     → Relación clásica entre cantidad y calidad de señal

6. **Ranking práctico**

   1. `t2_p40_h30` (mejor equilibrio)
   2. `t2_p40_h60`
   3. `t2_p50_h30` (más conservador)

7. **Conclusión**

   * Targets bien construidos
   * Señal real presente
   * Dataset apto para modelado

   → Existe estructura explotable en el target


### **Verificación de balance por clase**

In [48]:
import pandas as pd
import numpy as np

# ============================================================
# Configuración
# ============================================================
targets = [
    "t2_p40_h30",
    "t2_p40_h60",
    "t2_p50_h30",
]

# ============================================================
# Análisis de señal por régimen
# ============================================================
rows = []

for target in targets:
    df = mnq_train.dropna(subset=[target]).copy()

    for regime in [1, 2]:
        df_r = df[df["regime_id"] == regime].copy()

        n_total = len(df_r)

        counts = df_r[target].value_counts().to_dict()
        n_short = counts.get(-1, 0)
        n_flat  = counts.get(0, 0)
        n_long  = counts.get(1, 0)

        pct_short = n_short / n_total if n_total > 0 else np.nan
        pct_flat  = n_flat  / n_total if n_total > 0 else np.nan
        pct_long  = n_long  / n_total if n_total > 0 else np.nan

        signal_mask = df_r[target] != 0
        signal_rate = signal_mask.mean()

        # Consistencia diaria
        daily_signal = (
            df_r.assign(signal=signal_mask)
                .groupby("date")["signal"]
                .mean()
        )

        rows.append({
            "target": target,
            "regime": regime,
            "n_total": n_total,
            "pct_short": pct_short,
            "pct_flat": pct_flat,
            "pct_long": pct_long,
            "signal_rate": signal_rate,
            "daily_mean": daily_signal.mean(),
            "daily_std": daily_signal.std(),
            "daily_cv": daily_signal.std() / daily_signal.mean() if daily_signal.mean() > 0 else np.nan,
        })

df_regime_signal = pd.DataFrame(rows)

# ============================================================
# Mostrar resultados
# ============================================================
pd.set_option("display.max_columns", None)

print("=" * 100)
print("SEÑAL POR RÉGIMEN")
print("=" * 100)
print(df_regime_signal)

print("\n" + "=" * 100)
print("RESUMEN COMPARATIVO")
print("=" * 100)

print(
    df_regime_signal[
        [
            "target",
            "regime",
            "pct_short",
            "pct_flat",
            "pct_long",
            "signal_rate",
            "daily_mean",
            "daily_std",
            "daily_cv",
        ]
    ].to_string(index=False, float_format=lambda x: f"{x:.4f}")
)

SEÑAL POR RÉGIMEN
       target  regime  n_total  pct_short  pct_flat  pct_long  signal_rate  \
0  t2_p40_h30       1    62220   0.233446  0.523320  0.243234     0.476680   
1  t2_p40_h30       2    62220   0.310222  0.341369  0.348409     0.658631   
2  t2_p40_h60       1    62220   0.266201  0.448361  0.285439     0.551639   
3  t2_p40_h60       2    62220   0.273899  0.420508  0.305593     0.579492   
4  t2_p50_h30       1    62220   0.183510  0.626294  0.190196     0.373706   
5  t2_p50_h30       2    62220   0.266972  0.439585  0.293443     0.560415   

   daily_mean  daily_std  daily_cv  
0    0.476680   0.230681  0.483934  
1    0.658631   0.210696  0.319900  
2    0.551639   0.299523  0.542969  
3    0.579492   0.290953  0.502082  
4    0.373706   0.232883  0.623171  
5    0.560415   0.237555  0.423892  

RESUMEN COMPARATIVO
    target  regime  pct_short  pct_flat  pct_long  signal_rate  daily_mean  daily_std  daily_cv
t2_p40_h30       1     0.2334    0.5233    0.2432       0.4

**Análisis resumido de señal por régimen**

1. Resultado principal
   El régimen 2 (opening) presenta una señal claramente superior al régimen 1 (premarket).

2. Comparación directa

Señal (signal_rate)

* t2_p40_h30: 0.47 vs 0.65
* t2_p40_h60: 0.55 vs 0.57
* t2_p50_h30: 0.37 vs 0.56

El régimen 2 tiene mayor frecuencia de señales y mejor operabilidad.

Balance de clases

* Régimen 1: predominio de la clase flat → menor actividad del mercado
* Régimen 2: distribución más equilibrada → mayor direccionalidad

Estabilidad (daily_cv)

* Régimen 2 muestra menor variabilidad y mayor consistencia en todos los targets

3. Insight
   La mayor parte de la señal explotable se concentra en el opening, mientras que el premarket presenta más ruido y menor direccionalidad.

4. Implicación para el modelo
   No es recomendable mezclar ambos regímenes en un único modelo.
   Opciones:

* Usar únicamente régimen 2
* Entrenar modelos separados por régimen

5. Ranking práctico
6. régimen 2 + t2_p40_h30
7. régimen 2 + t2_p40_h60
8. régimen 2 + t2_p50_h30

El régimen 1 queda claramente por detrás.

6. Conclusión

* Existe señal real
* Está concentrada en régimen 2
* Régimen 1 aporta poco valor

Decisión recomendada: entrenar el modelo utilizando únicamente el régimen 2.


## **5.1. Decisión operativa: Filtrado solo de regimen 2 'Opening'**

In [49]:
mnq_r2 = mnq_model[mnq_model["regime_id"] == 2].copy()

In [52]:
mnq_r2.isna().any().any()

np.False_

In [50]:
mnq_r2_train, mnq_r2_valid, mnq_r2_test = time_based_split_by_date(
    mnq_r2,
    date_col="date",
    train_ratio=0.70,
    valid_ratio=0.15,
    test_ratio=0.15,
    verbose=True,
)

TIME SPLIT SUMMARY
Total días        : 1482
Train días        : 1037 (69.97%)
Valid días        : 222 (14.98%)
Test días         : 223 (15.05%)
--------------------------------------------------------------------------------
Train fechas      : 2020-01-02 → 2024-05-13
Valid fechas      : 2024-05-14 → 2025-05-06
Test fechas       : 2025-05-07 → 2026-04-17
--------------------------------------------------------------------------------
Train filas       : 62,220
Valid filas       : 13,320
Test filas        : 13,380


## **5.2. Evaluación multiclass balanceado**

In [55]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


def evaluate_multiclass_classifier_balanced(
    *,
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str],
    model_name: str = "logistic",
    impute_strategy: str = "median",
    scale_features: bool = True,
    dropna_target: bool = True,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """
    Entrena un clasificador multiclase baseline con class_weight="balanced".

    Targets esperados:
        -1, 0, 1

    Naive:
        siempre predice la clase mayoritaria del train

    Métricas:
        - accuracy
        - balanced_accuracy
        - f1_macro
    """

    # -----------------------------
    # Validaciones
    # -----------------------------
    missing_features = [c for c in feature_cols if c not in train_df.columns]
    if missing_features:
        raise ValueError(f"Faltan features en train_df: {missing_features}")

    for tgt in target_cols:
        for split_name, split_df in {
            "train": train_df,
            "valid": valid_df,
            "test": test_df,
        }.items():
            if tgt not in split_df.columns:
                raise ValueError(f"Falta target '{tgt}' en split '{split_name}'")

    # -----------------------------
    # Modelo
    # -----------------------------
    if model_name == "logistic":
        clf = LogisticRegression(
            max_iter=3000,
            multi_class="auto",
            random_state=42,
            class_weight="balanced",  # 🔥 CAMBIO CLAVE
        )
    elif model_name == "ridge":
        clf = RidgeClassifier(
            random_state=42,
            class_weight="balanced",  # 🔥 CAMBIO CLAVE
        )
    else:
        raise ValueError("model_name debe ser 'logistic' o 'ridge'")

    steps = [("imputer", SimpleImputer(strategy=impute_strategy))]
    if scale_features:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", clf))
    pipeline = Pipeline(steps)

    split_map = {
        "train": train_df,
        "valid": valid_df,
        "test": test_df,
    }

    results = []
    fitted_models = {}

    # -----------------------------
    # Loop por target
    # -----------------------------
    for target_col in target_cols:
        train_work = train_df[feature_cols + [target_col]].copy()
        if dropna_target:
            train_work = train_work[train_work[target_col].notna()].copy()

        X_train = train_work[feature_cols]
        y_train = train_work[target_col].astype(int)

        train_classes = sorted(y_train.unique().tolist())
        if len(train_classes) < 2:
            raise ValueError(
                f"El target '{target_col}' no tiene suficientes clases: {train_classes}"
            )

        majority_class = int(y_train.value_counts().idxmax())

        model = pipeline.fit(X_train, y_train)
        fitted_models[target_col] = {
            "model": model,
            "majority_class": majority_class,
        }

        # -----------------------------
        # Evaluación
        # -----------------------------
        for split_name, split_df in split_map.items():
            work = split_df[feature_cols + [target_col]].copy()
            if dropna_target:
                work = work[work[target_col].notna()].copy()

            X = work[feature_cols]
            y = work[target_col].astype(int)

            y_pred = model.predict(X)
            y_pred_naive = np.full(len(y), majority_class)

            acc_model = accuracy_score(y, y_pred)
            bal_acc_model = balanced_accuracy_score(y, y_pred)
            f1_model = f1_score(y, y_pred, average="macro", zero_division=0)

            acc_naive = accuracy_score(y, y_pred_naive)
            bal_acc_naive = balanced_accuracy_score(y, y_pred_naive)
            f1_naive = f1_score(y, y_pred_naive, average="macro", zero_division=0)

            class_dist = y.value_counts(normalize=True).to_dict()

            results.append(
                {
                    "model": model_name + "_balanced",
                    "target": target_col,
                    "split": split_name,
                    "n_samples": len(y),

                    "pct_-1": class_dist.get(-1, 0.0),
                    "pct_0": class_dist.get(0, 0.0),
                    "pct_1": class_dist.get(1, 0.0),

                    "acc_model": acc_model,
                    "acc_naive": acc_naive,
                    "acc_gain": acc_model - acc_naive,

                    "bal_acc_model": bal_acc_model,
                    "bal_acc_naive": bal_acc_naive,
                    "bal_acc_gain": bal_acc_model - bal_acc_naive,

                    "f1_model": f1_model,
                    "f1_naive": f1_naive,
                    "f1_gain": f1_model - f1_naive,
                }
            )

    results_df = pd.DataFrame(results)

    if verbose:
        print("=" * 120)
        print(f"BASELINE T2 (BALANCED) | model={model_name}")
        print("=" * 120)

        cols_show = [
            "target",
            "split",
            "acc_model",
            "acc_naive",
            "acc_gain",
            "bal_acc_model",
            "bal_acc_naive",
            "bal_acc_gain",
            "f1_model",
            "f1_naive",
            "f1_gain",
        ]

        print(results_df[cols_show].round(4).to_string(index=False))

    return results_df, fitted_models

In [56]:
features_core

['roc_60', 'ema_60', 'roc_30', 'stoch_k_30', 'regime_id']

In [62]:
target_cols = [
    "t2_p40_h30",
    "t2_p40_h60",
    "t2_p50_h30",
]

print('Con features_core')

results_baseline_core, models_baseline_core = evaluate_multiclass_classifier_balanced(
    train_df=mnq_r2_train,
    valid_df=mnq_r2_valid,
    test_df=mnq_r2_test,
    feature_cols=features_core,
    target_cols=target_cols,
    model_name="logistic",  # o "ridge"
    verbose=True,
)

Con features_core
BASELINE T2 (BALANCED) | model=logistic
    target split  acc_model  acc_naive  acc_gain  bal_acc_model  bal_acc_naive  bal_acc_gain  f1_model  f1_naive  f1_gain
t2_p40_h30 train     0.3527     0.3484    0.0043         0.3578         0.3333        0.0245    0.3230    0.1723   0.1508
t2_p40_h30 valid     0.3412     0.4014   -0.0601         0.3736         0.3333        0.0403    0.3149    0.1909   0.1239
t2_p40_h30  test     0.3374     0.4047   -0.0673         0.3699         0.3333        0.0366    0.3082    0.1921   0.1161
t2_p40_h60 train     0.3774     0.4205   -0.0431         0.3673         0.3333        0.0339    0.3489    0.1974   0.1516
t2_p40_h60 valid     0.3655     0.3268    0.0387         0.3737         0.3333        0.0404    0.3383    0.1642   0.1741
t2_p40_h60  test     0.3620     0.3026    0.0593         0.3791         0.3333        0.0457    0.3370    0.1549   0.1821
t2_p50_h30 train     0.3787     0.4396   -0.0609         0.3591         0.3333        0.

In [63]:
print('Con features_extended')

results_baseline_ext, models_baseline_ext = evaluate_multiclass_classifier_balanced(
    train_df=mnq_r2_train,
    valid_df=mnq_r2_valid,
    test_df=mnq_r2_test,
    feature_cols=features_extended,
    target_cols=target_cols,
    model_name="logistic",  # o "ridge"
    verbose=True,
)

Con features_extended
BASELINE T2 (BALANCED) | model=logistic
    target split  acc_model  acc_naive  acc_gain  bal_acc_model  bal_acc_naive  bal_acc_gain  f1_model  f1_naive  f1_gain
t2_p40_h30 train     0.3988     0.3484    0.0504         0.3960         0.3333        0.0627    0.3804    0.1723   0.2081
t2_p40_h30 valid     0.3462     0.4014   -0.0552         0.3919         0.3333        0.0586    0.3321    0.1909   0.1412
t2_p40_h30  test     0.3428     0.4047   -0.0620         0.3909         0.3333        0.0576    0.3192    0.1921   0.1271
t2_p40_h60 train     0.4407     0.4205    0.0202         0.4104         0.3333        0.0771    0.4063    0.1974   0.2090
t2_p40_h60 valid     0.3923     0.3268    0.0655         0.3973         0.3333        0.0639    0.3640    0.1642   0.1998
t2_p40_h60  test     0.3897     0.3026    0.0871         0.4096         0.3333        0.0763    0.3529    0.1549   0.1980
t2_p50_h30 train     0.4406     0.4396    0.0010         0.4003         0.3333      

### **Comparación de resultados**

In [61]:
results_baseline_core["feature_set"] = "core"
results_baseline_ext["feature_set"] = "extended"

df_compare = pd.concat([
    results_baseline_core,
    results_baseline_ext
])

df_compare.sort_values(
    ["target", "split", "bal_acc_gain"],
    ascending=[True, True, False]
)

,model,target,split,n_samples,pct_-1,pct_0,pct_1,acc_model,acc_naive,acc_gain,bal_acc_model,bal_acc_naive,bal_acc_gain,f1_model,f1_naive,f1_gain,feature_set
2,logistic_balanced,t2_p40_h30,test,13380,0.338266,0.257025,0.404709,0.342750,0.404709,-0.061958,0.390925,0.333333,0.057591,0.319166,0.192072,0.127094,extended
2,logistic_balanced,t2_p40_h30,test,13380,0.338266,0.257025,0.404709,0.337369,0.404709,-0.067339,0.369897,0.333333,0.036564,0.308213,0.192072,0.116140,core
0,logistic_balanced,t2_p40_h30,train,62220,0.310222,0.341369,0.348409,0.398827,0.348409,0.050418,0.396038,0.333333,0.062704,0.380387,0.172257,0.208130,extended
0,logistic_balanced,t2_p40_h30,train,62220,0.310222,0.341369,0.348409,0.352716,0.348409,0.004307,0.357797,0.333333,0.024463,0.323042,0.172257,0.150785,core
1,logistic_balanced,t2_p40_h30,valid,13320,0.348724,0.249925,0.401351,0.346171,0.401351,-0.055180,0.391947,0.333333,0.058614,0.332088,0.190935,0.141152,extended
1,logistic_balanced,t2_p40_h30,valid,13320,0.348724,0.249925,0.401351,0.341216,0.401351,-0.060135,0.373609,0.333333,0.040276,0.314851,0.190935,0.123915,core
5,logistic_balanced,t2_p40_h60,test,13380,0.321749,0.302616,0.375635,0.389686,0.302616,0.087070,0.409607,0.333333,0.076273,0.352888,0.154876,0.198012,extended
5,logistic_balanced,t2_p40_h60,test,13380,0.321749,0.302616,0.375635,0.361958,0.302616,0.059342,0.379060,0.333333,0.045726,0.336978,0.154876,0.182102,core
3,logistic_balanced,t2_p40_h60,train,62220,0.273899,0.420508,0.305593,0.440710,0.420508,0.020203,0.410391,0.333333,0.077058,0.406340,0.197351,0.208989,extended
3,logistic_balanced,t2_p40_h60,train,62220,0.273899,0.420508,0.305593,0.377387,0.420508,-0.043121,0.367258,0.333333,0.033924,0.348936,0.197351,0.151585,core


## **5.3. Conclusión parcial — Evaluación de targets T2 (clasificación multiclase)**

**Resultado principal**

Las features extendidas son claramente superiores a las features core en todos los targets evaluados.
Se observa de forma consistente:

* mayor `bal_acc_gain`
* mayor `f1_gain`
* estabilidad en train, valid y test

---

**Evidencia**

Para el target t2_p40_h60:

* TEST:

  * extended: `bal_acc_gain = 0.076`
  * core: `bal_acc_gain = 0.046`

Esto representa una mejora relevante de aproximadamente 0.03, lo que indica un aporte real de las features adicionales.

---

**Overfitting**

No se observan signos de overfitting.

* VALID y TEST son consistentes
* En algunos casos, TEST incluso supera VALID

Ejemplo t2_p40_h60:

* valid: 0.063
* test: 0.076

Esto indica buena capacidad de generalización.

---

**Evaluación por target**

- `t2_p40_h30`

  * `bal_acc_gain` ~ 0.05–0.06
  * señal moderada

- `t2_p40_h60`

  * `bal_acc_gain` ~ 0.06–0.08
  * mejor desempeño general

- `t2_p50_h30`

  * `bal_acc_gain` ~ 0.05–0.06
  * más conservador, menor frecuencia

---

**Insight clave**

El modelo logístico, sin tuning ni complejidad adicional, ya captura señal relevante.

Esto implica que el problema es aprendible y que la señal no depende de modelos complejos.

---

**Interpretación**

Los resultados confirman que:

* las features extendidas agregan información útil
* no introducen ruido significativo
* no degradan el desempeño fuera de muestra

---

**Ranking**

1. `t2_p40_h60` (extended)
2. `t2_p50_h30` (extended)
3. `t2_p40_h30` (extended)

---

**Conclusión**

* Existe señal explotable
* Las features extendidas aportan valor
* El modelo generaliza correctamente

Decisión recomendada: avanzar con features extendidas y el `target t2_p40_h60`.

# **8. Guardado de datasets**


## **8.1. Definición de rutas**

In [87]:
#Sin regime_id
features_cols = ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']

In [88]:
target_cols

['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']

In [89]:
time_cols = [
    "date",
    "minute_of_day",
]

In [90]:
print('Columnas a mantener:')
print(f'time_cols: {time_cols}')
print(f'features_cols: {features_cols}')
print(f'target_cols: {target_cols}')

Columnas a mantener:
time_cols: ['date', 'minute_of_day']
features_cols: ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
target_cols: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']


In [91]:
cols_final = time_cols + features_cols + target_cols

mnq_t2 = mnq_r2[cols_final].copy()

In [92]:
mnq_t2

,date,minute_of_day,ema_60,roc_60,roc_30,stoch_k_30,mom_5,atr_norm_10,macd,t2_p40_h30,t2_p40_h60,t2_p50_h30
datetime,,,,,,,,,,,,
2020-01-02 09:30:00-05:00,2020-01-02,570,-0.000288,-0.022669,-0.087786,35.714286,-0.000057,0.000302,-0.069907,1.0,0.0,1.0
2020-01-02 09:31:00-05:00,2020-01-02,571,0.000324,0.042506,-0.022655,82.352941,0.000765,0.000377,0.328333,0.0,0.0,0.0
2020-01-02 09:32:00-05:00,2020-01-02,572,-0.000481,-0.056665,-0.082144,25.490196,-0.000198,0.000461,0.107615,1.0,0.0,0.0
2020-01-02 09:33:00-05:00,2020-01-02,573,-0.000136,-0.045322,-0.002834,49.019608,0.000312,0.000480,0.164403,0.0,0.0,0.0
2020-01-02 09:34:00-05:00,2020-01-02,574,0.000937,0.062316,0.090667,90.140845,0.001162,0.000570,0.822544,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-17 10:25:00-04:00,2026-04-17,625,0.000304,-0.044859,0.102016,58.309859,-0.000271,0.000768,-4.477908,1.0,1.0,1.0
2026-04-17 10:26:00-04:00,2026-04-17,626,0.000285,-0.000935,0.135756,58.028169,-0.000103,0.000733,-4.186233,1.0,1.0,1.0
2026-04-17 10:27:00-04:00,2026-04-17,627,0.000095,0.060822,0.108596,49.702381,-0.000645,0.000731,-4.152592,1.0,1.0,1.0


In [93]:
OUT_PARQUET = DRIVE_DIR / Path(os.environ.get("OUT_PARQUET", "data/04_features/mnq_t2.parquet"))
OUT_SUMMARY = DRIVE_DIR /Path(os.environ.get("OUT_SUMMARY", "data/04_features/mnq_t2_summary.json"))

## **8.2. Guardado de dataset**

In [94]:
for path, df in [
    (OUT_PARQUET, mnq_t2),
    ]:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=True)
    print(f"[OK] Guardado: {path}")

[OK] Guardado: /content/drive/MyDrive/neural_profit/data/04_features/mnq_t2.parquet


## **8.3. Generación y guardado de summary**

In [98]:
import json
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# 1. Función summary corregida
# ============================================================
def build_mnq_t2_summary(
    df: pd.DataFrame,
    *,
    out_json_path: str | Path,
    aux_cols: list[str],
    feature_cols: list[str],
    target_cols: list[str],
    date_col: str = "date",
    regime_col: str = "regime_id",
    verbose: bool = True,
) -> dict:
    """
    Genera un summary completo del dataset mnq_t2 y lo guarda en JSON.

    Incluye:
    - shape y columnas
    - rango temporal
    - número de sesiones
    - NaNs por grupo de columnas
    - distribución de targets
    - distribución de regímenes
    - estadísticas básicas de variables numéricas
    """

    df = df.copy()

    # ------------------------------------------------------------------
    # Validaciones
    # ------------------------------------------------------------------
    required_cols = aux_cols + feature_cols + target_cols
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas en el dataset: {missing}")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser DatetimeIndex")

    if not df.index.is_monotonic_increasing:
        raise ValueError("El índice no está ordenado cronológicamente")

    # ------------------------------------------------------------------
    # Info general
    # ------------------------------------------------------------------
    summary = {}

    summary["shape"] = {
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
    }

    summary["time_range"] = {
        "start": str(df.index.min()),
        "end": str(df.index.max()),
    }

    summary["sessions"] = {
        "n_sessions": int(df[date_col].nunique()),
    }

    # ------------------------------------------------------------------
    # Column groups
    # ------------------------------------------------------------------
    summary["column_groups"] = {
        "auxiliary": aux_cols,
        "features": feature_cols,
        "targets": target_cols,
    }

    # ------------------------------------------------------------------
    # NaNs
    # ------------------------------------------------------------------
    def _nan_report(cols: list[str]) -> dict:
        return {
            "n_nan_total": int(df[cols].isna().sum().sum()),
            "n_nan_by_col": {
                col: int(df[col].isna().sum()) for col in cols
            }
        }

    summary["nan_report"] = {
        "auxiliary": _nan_report(aux_cols),
        "features": _nan_report(feature_cols),
        "targets": _nan_report(target_cols),
    }

    # ------------------------------------------------------------------
    # Distribución de targets
    # ------------------------------------------------------------------
    target_distribution = {}

    for tgt in target_cols:
        vc = df[tgt].value_counts(dropna=False).sort_index()
        total = vc.sum()

        target_distribution[tgt] = {
            str(int(k)) if pd.notna(k) else "nan": {
                "count": int(v),
                "pct": float(v / total) if total > 0 else np.nan,
            }
            for k, v in vc.items()
        }

    summary["target_distribution"] = target_distribution

    # ------------------------------------------------------------------
    # Distribución de régimen
    # ------------------------------------------------------------------
    regime_name_map = {
        0: "overnight",
        1: "premarket",
        2: "opening",
        3: "regular",
        4: "closing",
    }

    vc_regime = df[regime_col].value_counts().sort_index()
    total_regime = vc_regime.sum()

    summary["regime_distribution"] = {
        int(k): {
            "name": regime_name_map.get(int(k), "unknown"),
            "count": int(v),
            "pct": float(v / total_regime),
        }
        for k, v in vc_regime.items()
    }

    # ------------------------------------------------------------------
    # Estadísticas numéricas
    # ------------------------------------------------------------------
    numeric_cols = feature_cols + [
        c for c in aux_cols if "delta" in c
    ]

    # eliminar duplicados por si alguna columna se repite
    numeric_cols = list(dict.fromkeys(numeric_cols))

    stats = {}

    for col in numeric_cols:
        if col not in df.columns:
            continue

        series = df[col].dropna()
        if len(series) == 0:
            continue

        desc = series.describe()

        stats[col] = {
            "mean": float(desc["mean"]),
            "std": float(desc["std"]),
            "min": float(desc["min"]),
            "p25": float(desc["25%"]),
            "p50": float(desc["50%"]),
            "p75": float(desc["75%"]),
            "max": float(desc["max"]),
        }

    summary["numeric_stats"] = stats

    # ------------------------------------------------------------------
    # Guardar JSON
    # ------------------------------------------------------------------
    out_json_path = Path(out_json_path)
    out_json_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4, ensure_ascii=False)

    if verbose:
        print("=" * 100)
        print("SUMMARY MNQ_T2 GENERADO")
        print("=" * 100)
        print(f"Filas                : {summary['shape']['n_rows']:,}")
        print(f"Columnas             : {summary['shape']['n_cols']:,}")
        print(f"Sesiones             : {summary['sessions']['n_sessions']:,}")
        print(f"Desde                : {summary['time_range']['start']}")
        print(f"Hasta                : {summary['time_range']['end']}")
        print(f"Archivo              : {out_json_path}")
        print("=" * 100)

    return summary


# ============================================================
# 2. Definir columnas
# ============================================================
# Mantener regime_id como auxiliar, no como feature
time_cols = [
    "date",
    "minute_of_day",
    "regime_id",
]

# Remover regime_id de las features si aún estuviera incluido
features_cols = [c for c in features_cols if c != "regime_id"]

target_cols = [
    "t2_p40_h30",
    "t2_p40_h60",
    "t2_p50_h30",
]

print("Columnas a mantener:")
print(f"time_cols     : {time_cols}")
print(f"features_cols : {features_cols}")
print(f"target_cols   : {target_cols}")


# ============================================================
# 3. Construir dataset final en el orden deseado
# ============================================================
cols_final = time_cols + features_cols + target_cols

missing_final = [c for c in cols_final if c not in mnq_r2.columns]
if missing_final:
    raise ValueError(f"Faltan columnas en mnq_r2: {missing_final}")

mnq_final = mnq_r2[cols_final].copy()

# Asegurar orden cronológico
mnq_final = mnq_final.sort_values(["date", "minute_of_day"]).copy()

# Mantener índice limpio y ordenado
mnq_final = mnq_final.sort_index().copy()

print("=" * 100)
print("DATASET FINAL CONSTRUIDO")
print("=" * 100)
print(f"Shape         : {mnq_final.shape}")
print(f"Columnas      : {mnq_final.columns.tolist()}")
print("=" * 100)


# ============================================================
# 4. Validaciones finales
# ============================================================
assert isinstance(mnq_final.index, pd.DatetimeIndex), \
    "El índice debe ser DatetimeIndex"

assert mnq_final.index.is_monotonic_increasing, \
    "El índice debe estar ordenado crecientemente"

assert not mnq_final[features_cols].isna().any().any(), \
    "Las features contienen NaN"

assert not mnq_final[target_cols].isna().any().any(), \
    "Los targets contienen NaN"

assert set(mnq_final["regime_id"].unique()) == {2}, \
    "El dataset final debería contener solo regime_id = 2"


# ============================================================
# 5. Generar summary
# ============================================================
OUT_JSON_PATH = "data/04_features/mnq_r2_summary.json"

summary = build_mnq_t2_summary(
    df=mnq_final,
    out_json_path=OUT_JSON_PATH,
    aux_cols=time_cols,
    feature_cols=features_cols,
    target_cols=target_cols,
    date_col="date",
    regime_col="regime_id",
    verbose=True,
)

Columnas a mantener:
time_cols     : ['date', 'minute_of_day', 'regime_id']
features_cols : ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
target_cols   : ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
DATASET FINAL CONSTRUIDO
Shape         : (88920, 13)
Columnas      : ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
SUMMARY MNQ_T2 GENERADO
Filas                : 88,920
Columnas             : 13
Sesiones             : 1,482
Desde                : 2020-01-02 09:30:00-05:00
Hasta                : 2026-04-17 10:29:00-04:00
Archivo              : data/04_features/mnq_r2_summary.json
